In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack

# 1. Chargement et Nettoyage
df = pd.read_csv("data/TMDB_IMDB_MoviesDataset.csv")

# Sélection des colonnes stratégiques
num_cols = ['vote_average', 'popularity', 'runtime', 'budget', 'revenue', 'averageRating', 'numVotes']
df = df[num_cols + ['genres', 'overview', 'release_date']].dropna()

# 2. Stratégie "Haute Précision" : Top 5 genres les plus discriminants
# On retire les genres trop ambigus pour monter vers 80-90%
top_genres = ['Drama', 'Comedy', 'Documentary', 'Action', 'Horror']

def filter_genres(g_str):
    genres = [g.strip() for g in str(g_str).split(',') if g.strip() in top_genres]
    # Pour booster l'accuracy, on se concentre sur les films ayant 1 ou 2 genres max du top
    return genres if 0 < len(genres) <= 2 else None

df['genres_list'] = df['genres'].apply(filter_genres)
df = df.dropna(subset=['genres_list'])

# 3. Équilibrage intelligent (Undersampling)
# On limite le Drama et le Documentaire qui sont souvent trop représentés
n_samples = min(len(df[df['genres'].str.contains('Drama')]), 8000)
df_drama = df[df['genres'].str.contains('Drama')].sample(n_samples, random_state=42)
df_others = df[~df['genres'].str.contains('Drama')]
df_final = pd.concat([df_drama, df_others])

# 4. Traitement du Texte (X1)
# On augmente max_features pour capturer le vocabulaire spécifique
tfidf = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1, 2))
X_text = tfidf.fit_transform(df_final['overview'])

# 5. Traitement Numérique Discrétisé (X2)
# BernoulliNB ne comprend pas les nombres continus, on les transforme en "mots" binaires
df_final['year'] = pd.to_datetime(df_final['release_date'], errors='coerce').dt.year
X_num_raw = df_final[num_cols + ['year']].fillna(0)

kbd = KBinsDiscretizer(n_bins=15, encode='onehot-dense', strategy='quantile')
X_num_bin = kbd.fit_transform(X_num_raw)

# 6. Fusion des Features (Texte + Numérique)
X_combined = hstack([X_text, X_num_bin])

# 7. Préparation de la cible
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df_final['genres_list'])

# 8. Split et Modèle
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.15, random_state=42)

# Alpha très bas pour coller aux données TF-IDF
model = OneVsRestClassifier(BernoulliNB(alpha=0.01))
model.fit(X_train, y_train)

# 9. Évaluation
y_pred = model.predict(X_test)

print(f"--- MODÈLE HYBRIDE (Texte + Numérique) ---")
print(f"Nombre de samples : {len(df_final)}")
print(f"Accuracy (Exact Match) : {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report :")
print(classification_report(y_test, y_pred, target_names=mlb.classes_))

fairouz filtre 80

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack

# --- TA FONCTION DE RÉDUCTION (ÉQUILIBRAGE) ---
def reduce_top_classes(df, target_col, n_classes=2, max_samples=10000, random_state=42):
    top_classes = df[target_col].value_counts().head(n_classes).index
    df_top = df[df[target_col].isin(top_classes)]
    df_rest = df[~df[target_col].isin(top_classes)]
    
    df_top_reduced = df_top.groupby(target_col).apply(
        lambda x: x.sample(n=min(len(x), max_samples), random_state=random_state)
    ).reset_index(drop=True)
    
    return pd.concat([df_top_reduced, df_rest]).reset_index(drop=True)

# 1. Chargement et Nettoyage de base
df = pd.read_csv("data/TMDB_IMDB_MoviesDataset.csv")
num_cols = ['vote_average', 'popularity', 'runtime', 'budget', 'revenue', 'averageRating', 'numVotes']
df = df[num_cols + ['genres', 'overview', 'release_date']].dropna()

# 2. LOGIQUE DES 80% DURCIE (Seuil à 0.55 pour réduire le nombre de genres)
all_genres = df['genres'].str.split(',').explode().str.strip()
genre_counts = all_genres.value_counts(normalize=True)
cumulative = genre_counts.cumsum()

# En baissant à 0.55, on ne garde que les 5-6 genres les plus massifs
top_genres_55 = cumulative[cumulative <= 0.55].index.tolist()

print(f"Genres ultra-majoritaires sélectionnés : {top_genres_55}")

def filter_genres_logic(g_str):
    # On filtre les genres par rapport au top 55%
    genres = [g.strip() for g in str(g_str).split(',') if g.strip() in top_genres_55]
    
    # STRATÉGIE ACCURACY : On limite à 2 genres maximum par film
    # Cela permet d'augmenter drastiquement l'Exact Match
    return genres[:5] if len(genres) > 0 else None

df['genres_list'] = df['genres'].apply(filter_genres_logic)
df = df.dropna(subset=['genres_list'])

# Création de la colonne pour l'équilibrage
df['main_genre'] = df['genres_list'].apply(lambda x: x[0])

# 3. Application de la réduction (Undersampling des genres dominants)
df_final = reduce_top_classes(df, 'main_genre', n_classes=2, max_samples=8000)

# 4. Traitement du Texte (X1)
# ngram_range(1,2) aide à distinguer "science fiction" de "fiction"
tfidf = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1, 2))
X_text = tfidf.fit_transform(df_final['overview'])

# 5. Traitement Numérique Discrétisé (X2)
df_final['year'] = pd.to_datetime(df_final['release_date'], errors='coerce').dt.year
X_num_raw = df_final[num_cols + ['year']].fillna(0)

# Discrétisation en 15 tranches (Secret de BernoulliNB pour le numérique)
kbd = KBinsDiscretizer(n_bins=15, encode='onehot-dense', strategy='quantile')
X_num_bin = kbd.fit_transform(X_num_raw)

# 6. Fusion des Features (Hybride)
X_combined = hstack([X_text, X_num_bin])

# 7. Préparation de la cible Multi-Label
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df_final['genres_list'])

# 8. Split et Modèle
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.15, random_state=42)

# Alpha très bas (0.01) pour une sensibilité maximale aux mots-clés
model = OneVsRestClassifier(BernoulliNB(alpha=0.01))
model.fit(X_train, y_train)

# 9. Évaluation
y_pred = model.predict(X_test)

print(f"\n--- RÉSULTATS MODÈLE HAUTE PRÉCISION ---")
print(f"Nombre de genres à prédire : {len(mlb.classes_)}")
print(f"Accuracy (Exact Match) : {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report :")
print(classification_report(y_test, y_pred, target_names=mlb.classes_))